# 🔓 Solutions: Supervised Learning

> **Try first, peek second.** You learn far more by struggling with a problem for ten minutes than by reading its answer for ten seconds. Use these solutions to *compare* your approach, not to replace it. There is often more than one correct way.


In [1]:
import sys
sys.path.append("..")
from course_utils import *
setup()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

### 🧩 Problem 2.1: Regression or classification? &nbsp; ⭐

Write a function `task_type(series)` that decides whether a target column calls for **`"regression"`** or **`"classification"`**.

Use this simple heuristic:
* text (object/string) column → `"classification"`
* numeric column with **10 or fewer** distinct values → `"classification"`
* otherwise → `"regression"`

In [2]:
def task_type(series):
    if not pd.api.types.is_numeric_dtype(series):
        return "classification"
    return "classification" if series.nunique() <= 10 else "regression"

print(task_type(load("students.csv")["exam_score"]))

regression


In [3]:
# ✔ Self-check
cases = [(load("students.csv")["exam_score"], "regression"),
         (load("churn.csv")["churned"], "classification"),
         (load("iris.csv")["species"], "classification"),
         (load("housing.csv")["price_k"], "regression"),
         (load("wine.csv")["target"], "classification"),
         (load("diabetes.csv")["disease_progression"], "regression")]
for s, expected in cases:
    check(f"{s.name:<22} -> {expected}", lambda: task_type(s) == expected)

✅ PASSED   exam_score             -> regression
✅ PASSED   churned                -> classification
✅ PASSED   species                -> classification
✅ PASSED   price_k                -> regression
✅ PASSED   target                 -> classification
✅ PASSED   disease_progression    -> regression


**🧑‍🏫 Walkthrough.** Text can't be averaged, so it must be a category. A numeric column with very few distinct values is almost certainly a category encoded as numbers (0/1, 1/2/3). Everything else is a quantity. **This is only a heuristic**: a "star rating" from 1–5 is numeric with five values but you might reasonably treat it as regression. In practice you decide by *what the label means*, not by its dtype.

### 🧩 Problem 2.2: The price of space &nbsp; ⭐

Using `housing.csv`, fit a **simple linear regression** of `price_k` on `area_sqm` only.

Store the slope (price change in $1000s per extra m²) in a variable called `slope_per_sqm`, and the predicted price of a **100 m²** house in `price_100`.

In [4]:
housing = load("housing.csv")

model = LinearRegression().fit(housing[["area_sqm"]], housing["price_k"])
slope_per_sqm = model.coef_[0]
price_100 = model.predict(pd.DataFrame({"area_sqm": [100]}))[0]
print(f"slope = {slope_per_sqm:.2f} k$/m²,  100 m² house ≈ {price_100:.0f} k$")

slope = 3.90 k$/m²,  100 m² house ≈ 405 k$


In [5]:
# ✔ Self-check
check("slope is positive and in a sensible range", lambda: 3.0 < slope_per_sqm < 6.5)
check("100 m² house is predicted between 200 and 450 ($1000s)", lambda: 200 < price_100 < 450)

✅ PASSED   slope is positive and in a sensible range
✅ PASSED   100 m² house is predicted between 200 and 450 ($1000s)


True

**🧑‍🏫 Walkthrough.** Each extra square metre is associated with roughly the slope's worth of thousands of dollars. Note that the slope here (≈ 3.9) is *larger* than the 2.4 built into the generating formula for area alone, because area also drives price through other terms (bigger houses tend to be in the same pricing bands, area interacts with quality, etc.). That's a first hint that **single-feature models absorb the effect of features they can't see**.

### 🧩 Problem 2.3: Price my house (multiple regression) &nbsp; ⭐⭐

Fit a multiple linear regression on `housing.csv` using these features:
`["area_sqm", "bathrooms", "age_years", "distance_to_center_km", "has_garden", "neighborhood_quality"]`

Store:
* `model`: the fitted model
* `coefs`: a pandas Series of its coefficients indexed by feature name
* `my_house_price`: the predicted price of a house with **area 140, bathrooms 2, age 10, distance 5, garden 1, quality 4**

In [6]:
housing = load("housing.csv")
features = ["area_sqm", "bathrooms", "age_years", "distance_to_center_km", "has_garden", "neighborhood_quality"]

model = LinearRegression().fit(housing[features], housing["price_k"])
coefs = pd.Series(model.coef_, index=features)
my_house = pd.DataFrame([[140, 2, 10, 5, 1, 4]], columns=features)
my_house_price = model.predict(my_house)[0]
print(coefs.round(2).to_string())
print(f"\nPredicted price: {my_house_price:.0f} k$")

area_sqm                  3.83
bathrooms                11.14
age_years                -1.73
distance_to_center_km    -4.51
has_garden               23.11
neighborhood_quality     47.51

Predicted price: 666 k$


In [7]:
# ✔ Self-check
check("age has a negative coefficient (older = cheaper)", lambda: coefs["age_years"] < 0)
check("garden has a positive coefficient", lambda: coefs["has_garden"] > 0)
check("price is plausible (450-900 $1000s)", lambda: 450 < my_house_price < 900)
check("price agrees with model.predict", lambda: abs(my_house_price - model.predict(pd.DataFrame([[140, 2, 10, 5, 1, 4]], columns=features))[0]) < 1e-6)

✅ PASSED   age has a negative coefficient (older = cheaper)
✅ PASSED   garden has a positive coefficient
✅ PASSED   price is plausible (450-900 $1000s)
✅ PASSED   price agrees with model.predict


True

**🧑‍🏫 Walkthrough.** Keeping a labelled `Series` of coefficients makes the model *readable*. And you can always double-check a prediction by hand: `model.intercept_ + (coefs * my_house.iloc[0]).sum()` equals `my_house_price`.

### 🧩 Problem 2.4: Name that flower (classification) &nbsp; ⭐⭐

Using `iris.csv`, train a `KNeighborsClassifier` with `n_neighbors=5` on **all four measurement columns** (`sepal_length_cm`, `sepal_width_cm`, `petal_length_cm`, `petal_width_cm`) to predict **`species`**.

Predict the species of these two flowers and store the two labels in a list `species_pred`:

| sepal length | sepal width | petal length | petal width |
|---|---|---|---|
| 5.1 | 3.5 | 1.4 | 0.2 |
| 6.7 | 3.0 | 5.2 | 2.3 |

In [8]:
iris = load("iris.csv")
feat_cols = ["sepal_length_cm", "sepal_width_cm", "petal_length_cm", "petal_width_cm"]

knn = KNeighborsClassifier(n_neighbors=5).fit(iris[feat_cols], iris["species"])
new = pd.DataFrame([[5.1, 3.5, 1.4, 0.2], [6.7, 3.0, 5.2, 2.3]], columns=feat_cols)
species_pred = list(knn.predict(new))
print(species_pred)

['setosa', 'virginica']


In [9]:
# ✔ Self-check
check("two predictions", lambda: len(species_pred) == 2)
check("first flower is a setosa", lambda: species_pred[0] == "setosa")
check("second flower is a virginica", lambda: species_pred[1] == "virginica")

✅ PASSED   two predictions
✅ PASSED   first flower is a setosa
✅ PASSED   second flower is a virginica


True

**🧑‍🏫 Walkthrough.** Small petals → setosa; large petals → virginica. The first flower is actually the very first row of the dataset, so its answer is "known": a reminder that predicting on *training* examples is too easy to be informative (the topic of Lesson 4).

### 🧩 Problem 2.5: How sure is the model? (probabilities) &nbsp; ⭐⭐

Load `breast_cancer.csv`. The target column `target` is **1 = benign, 0 = malignant**; all columns except `target` and `diagnosis` are features.

1. Train `clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))` on all of it.
2. Compute the predicted probability of class **1 (benign)** for every tumour.
3. Find the row whose probability is **closest to 0.5**: the tumour the model is *least sure about*. Store its position in `most_uncertain_idx`.

In [10]:
bc = load("breast_cancer.csv")
X = bc.drop(columns=["target", "diagnosis"])
y = bc["target"]

clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X, y)
p_benign = clf.predict_proba(X)[:, 1]
most_uncertain_idx = int(np.argmin(np.abs(p_benign - 0.5)))
print(f"row {most_uncertain_idx}: P(benign) = {p_benign[most_uncertain_idx]:.3f}")

row 541: P(benign) = 0.454


In [11]:
# ✔ Self-check
check("classifier was trained", lambda: hasattr(clf, "predict_proba"))
check("found the most uncertain row", lambda: most_uncertain_idx == int(np.argmin(np.abs(clf.predict_proba(X)[:, 1] - 0.5))))
check("its probability is close to 0.5 (within 0.1)", lambda: abs(clf.predict_proba(X)[most_uncertain_idx, 1] - 0.5) < 0.1)

✅ PASSED   classifier was trained
✅ PASSED   found the most uncertain row
✅ PASSED   its probability is close to 0.5 (within 0.1)


True

**🧑‍🏫 Walkthrough.** `predict_proba` returns one column per class, in the order of `clf.classes_` (here `[0, 1]`). Taking `|p − 0.5|` measures how far each prediction is from a coin-flip. In real medicine this is exactly how a model "knows when to ask a human": send the *uncertain* cases to a doctor.

### 🧩 Problem 2.6: KNN from scratch &nbsp; ⭐⭐⭐

Implement `knn_predict(X_train, y_train, x_new, k)` **using only NumPy** (no sklearn inside the function):

1. compute the Euclidean distance from `x_new` to every row of `X_train`,
2. find the `k` smallest distances,
3. return the **most common label** among those `k` neighbours.

`X_train` is a 2-D NumPy array, `y_train` a 1-D array of **integer** labels, `x_new` a 1-D array.

In [12]:
def knn_predict(X_train, y_train, x_new, k=5):
    dist = np.sqrt(((X_train - x_new) ** 2).sum(axis=1))   # 1. distance to every training point
    nearest = np.argsort(dist)[:k]                          # 2. indices of the k closest
    return np.bincount(y_train[nearest]).argmax()           # 3. majority vote

iris = load("iris.csv")
Xa = iris[["sepal_length_cm", "sepal_width_cm", "petal_length_cm", "petal_width_cm"]].to_numpy()
ya = iris["target"].to_numpy()
print([knn_predict(Xa, ya, row, k=5) for row in Xa[:5]])

[np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0)]


In [13]:
# ✔ Self-check
iris = load("iris.csv")
Xa = iris[["sepal_length_cm", "sepal_width_cm", "petal_length_cm", "petal_width_cm"]].to_numpy()
ya = iris["target"].to_numpy()
reference = KNeighborsClassifier(n_neighbors=5).fit(Xa, ya)
mine = np.array([knn_predict(Xa, ya, row, k=5) for row in Xa[::5]])
agree = (mine == reference.predict(Xa[::5])).mean()
check(f"agrees with sklearn on {agree:.0%} of 30 flowers (need ≥ 95%)", lambda: agree >= 0.95)

✅ PASSED   agrees with sklearn on 100% of 30 flowers (need ≥ 95%)


True

**🧑‍🏫 Walkthrough.** Three lines of NumPy reproduce a scikit-learn model. Notice that **KNN has no real "training"**: `fit` just stores the data, and all the work happens at prediction time (that's why it's called a *lazy learner*). Other algorithms, like linear regression, do the opposite: heavy work in `fit`, trivial work in `predict`.